# Optimal Block Bootstrap Size Estimation via Subset Sampling
 
 This notebook estimates the optimal block size for 2D structure function bootstrap 
 by sampling multiple subsets and finding the scaling exponent k.
 
 **Method:**
 1. Randomly select N subsets from the full dataset
 2. For each subset, find the optimal bootsize via convergence testing
 3. Compute k = ln(n_subset) / ln(b_opt_subset) for each
 4. Take median(k) as the robust estimate
 5. Extrapolate to full dataset: b_opt_full = n_full^(1/k_median)


# Load Modules

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pyturbo_sf as psf
import warnings
import pyproj
from tqdm import tqdm
from scipy import stats 
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'serif'


# Configuration Parameters

In [2]:
# =============================================================================
# USER CONFIGURATION - MODIFY THESE AS NEEDED
# =============================================================================


# Full dataset dimensions
FULL_DIM_Y = 2820
FULL_DIM_X = 2000

# Multiple subset sizes for multi-scale regression (y, x)
# These should maintain similar aspect ratio to full dataset (~1.41)
option = 'multi-subset'

if option == 'multi-subset':
    
    SUBSET_SIZES = [
        (64, 45),
        (90, 64),
        (128, 90),
        (180, 128),
        (256, 180),
    ]


    # Number of random subsets to sample per size
    N_SUBSETS_PER_SIZE = 10

elif option == 'single-subset':

    SUBSET_SIZES = [
        (128, 90),
    ]


    # Number of random subsets to sample per size 
    N_SUBSETS_PER_SIZE = 100

# Maximum allowed NaN fraction in a subset (skip if exceeded)
MAX_NAN_FRACTION = 0.3

# Structure function bins
NBINS = 11
BIN_R = np.logspace(np.log10(1.0e3), np.log10(1.0e6), int(NBINS + 1))

# Bootstrap parameters that you can increase
INITIAL_NBOOTSTRAP = 1000
MAX_NBOOTSTRAP = 1000

# Confidence interval
CONFIDENCE_INTERVAL = 0.95

# Error threshold for optimal bootsize selection (%)
# Optimal = smallest bootsize with error below this threshold
ERROR_THRESHOLD = 5.0



# Load Dataset

In [3]:
idt = 330
dsg = xr.open_dataset('/home/aayouche/Downloads/FRAM_grid.nc')
dsu = xr.open_dataset('/home/aayouche/Downloads/FRAM_u.nc').isel(time=idt)
dsv = xr.open_dataset('/home/aayouche/Downloads/FRAM_v.nc').isel(time=idt)
dsm = xr.open_dataset('/home/aayouche/Downloads/FRAM_sic.nc').isel(time=idt)
u = dsu.U.rename({'j':'y','i':'x'})
v = dsv.V.rename({'j':'y','i':'x'})
sic = dsm.SIarea.rename({'j':'y','i':'x'})
lon = dsg.XC
lat = dsg.YC

# # Create projected coordinates
proj = pyproj.Proj("EPSG:3995")
x, y = proj(lon, lat)
y_dim, x_dim = x.shape

X = xr.DataArray(
    x,
    dims=['y', 'x'],
    coords={
        'y': np.arange(y_dim),
        'x': np.arange(x_dim)
    }
)

Y = xr.DataArray(
    y,
    dims=['y', 'x'],
    coords={
        'y': np.arange(y_dim),
        'x': np.arange(x_dim)
    }
)
# # Build dataset
ds = xr.Dataset(
     data_vars={'u': u, 'v': v, 'sic': sic},
     coords={'x': X, 'y': Y}
 )

print(f"Expected full dataset shape: y={FULL_DIM_Y}, x={FULL_DIM_X}")


Expected full dataset shape: y=2820, x=2000


# Helper Functions

In [4]:
def check_nan_fraction(ds_subset, variables=['u', 'v'], max_fraction=0.3):
    """Check if subset has acceptable NaN fraction."""
    total_nans = 0
    total_points = 0
    for var in variables:
        if var in ds_subset:
            data = ds_subset[var].values
            total_nans += np.sum(np.isnan(data))
            total_points += data.size
    
    if total_points == 0:
        return False
    
    nan_fraction = total_nans / total_points
    return nan_fraction <= max_fraction


def compute_k_from_bootsize(n_subset, b_opt):
    """
    Compute scaling exponent k from: b_opt = n^(1/k)
    
    Parameters:
    -----------
    n_subset : int
        Total number of points in subset (dim_y * dim_x)
    b_opt : int
        Optimal block size (area: b_y * b_x)
    
    Returns:
    --------
    k : float
        Scaling exponent
    """
    if b_opt <= 1:
        return np.nan
    k = np.log(n_subset) / np.log(b_opt)
    return k


def find_optimal_bootsize_for_subset(ds_subset, bins, base_bootsize_y, base_bootsize_x,
                                      variables_names=['u', 'v'], error_threshold=5.0):
    """
    Find optimal bootsize for a single subset by testing multiple sizes.
    
    Strategy: Find the SMALLEST bootsize that achieves mean relative error 
    below threshold compared to the "true" SF (computed with largest bootsize).
    This avoids the bias toward always picking the largest tested bootsize.
    
    Parameters:
    -----------
    error_threshold : float
        Maximum acceptable mean relative error (%). Default 5%.
    """
    # Generate bootsizes by dividing by powers of 2
    # Start from power=2 / leveraging bootstrapping efficiency
    max_power = int(np.log2(min(base_bootsize_x, base_bootsize_y)))
    powers = np.arange(2, max_power + 1)
    bootsizes_x = base_bootsize_x // (2 ** powers)
    bootsizes_y = base_bootsize_y // (2 ** powers)
    
    # Filter out sizes < 2
    valid_mask = (bootsizes_x >= 2) & (bootsizes_y >= 2)
    bootsizes_x = bootsizes_x[valid_mask]
    bootsizes_y = bootsizes_y[valid_mask]
    powers = powers[valid_mask]
    
    if len(bootsizes_x) == 0:
        return None, None, None
    
    # Base kwargs
    base_kwargs = dict(
        ds=ds_subset,
        variables_names=variables_names,
        bins=bins,
        fun='longitudinal',
        order=2,
        initial_nbootstrap=INITIAL_NBOOTSTRAP,
        max_nbootstrap=MAX_NBOOTSTRAP,
        step_nbootstrap=0,
        convergence_eps=0.001,
        confidence_interval=CONFIDENCE_INTERVAL,
        n_jobs=-1,
        backend='loky',
        seed=42
    )
    
    # Compute "true" SF with largest bootsize (full subset size)
    try:
        true_sf = psf.get_isotropic_sf_2d(
            **base_kwargs,
            bootsize={'y': base_bootsize_y, 'x': base_bootsize_x}
        )
        true_sf_values = true_sf['sf'].values
    except Exception as e:
        print(f"    Error computing true SF: {e}")
        return None, None, None
    
    # Test each bootsize from SMALLEST to LARGEST
    # (reverse order: highest power first = smallest bootsize first)
    results = {}
    for bsx, bsy, pwr in zip(bootsizes_x[::-1], bootsizes_y[::-1], powers[::-1]):
        try:
            sf_result = psf.get_isotropic_sf_2d(
                **base_kwargs,
                bootsize={'y': bsy, 'x': bsx}
            )
            sf_values = sf_result['sf'].values
            
            # Compute mean relative error
            rel_diff = np.abs(sf_values - true_sf_values) / np.abs(true_sf_values) * 100
            mean_rel_error = np.nanmean(rel_diff)
            
            results[(bsx, bsy)] = {
                'sf': sf_result,
                'mean_rel_error': mean_rel_error,
                'power': pwr
            }
        except Exception as e:
            continue
    
    if len(results) == 0:
        return None, None, None
    
    # Strategy: Find SMALLEST bootsize (highest power) with error < threshold
    # Sort by bootsize area (smallest first)
    sorted_bootsizes = sorted(results.keys(), key=lambda x: x[0] * x[1])
    
    best_bootsize = None
    for bs in sorted_bootsizes:
        if results[bs]['mean_rel_error'] < error_threshold:
            best_bootsize = bs
            break
    
    # If none below threshold, fall back to minimum error (but warn)
    if best_bootsize is None:
        best_bootsize = min(results.keys(), key=lambda x: results[x]['mean_rel_error'])
        print(f"    Warning: No bootsize below {error_threshold}% error, using min error")
    
    best_error = results[best_bootsize]['mean_rel_error']
    
    return best_bootsize, best_error, results


# Sample Dataset And Compute k values

In [5]:
def run_multiscale_analysis(ds, subset_sizes, n_subsets_per_size=10, error_threshold=5.0, seed=42):
    """
    Sample subsets of multiple sizes and compute optimal bootsize for each.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Full dataset with coordinates
    subset_sizes : list of tuples
        List of (dim_y, dim_x) subset sizes to test
    n_subsets_per_size : int
        Number of subsets to sample per size
    error_threshold : float
        Maximum acceptable mean relative error (%)
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    all_k_data : list of dicts
        Each entry contains individual subset info and computed k value
    """
    np.random.seed(seed)
    
    bins = {'r': BIN_R}
    all_k_data = []
    
    print(f"Multi-scale analysis with {len(subset_sizes)} subset sizes")
    print(f"Sampling {n_subsets_per_size} subsets per size")
    print(f"Total samples: {len(subset_sizes) * n_subsets_per_size}")
    print(f"Error threshold: {error_threshold}%")
    print("=" * 60)
    
    for dim_y, dim_x in subset_sizes:
        print(f"\nProcessing subset size: {dim_y} x {dim_x}")
        
        n_subset = dim_y * dim_x
        
        attempt = 0
        valid_count = 0
        max_attempts = n_subsets_per_size * 5
        
        pbar = tqdm(total=n_subsets_per_size, desc=f"  Size {dim_y}x{dim_x}")
        
        while valid_count < n_subsets_per_size and attempt < max_attempts:
            attempt += 1
            
            # Random starting indices
            idy = np.random.randint(0, FULL_DIM_Y - dim_y)
            idx = np.random.randint(0, FULL_DIM_X - dim_x)
            
            # Extract subset
            ds_subset = ds.isel(
                y=slice(idy, idy + dim_y),
                x=slice(idx, idx + dim_x)
            )
            
            # Check NaN fraction
            if not check_nan_fraction(ds_subset, max_fraction=MAX_NAN_FRACTION):
                continue
            
            # Find optimal bootsize
            best_bootsize, best_error, _ = find_optimal_bootsize_for_subset(
                ds_subset, bins, dim_y, dim_x, error_threshold=error_threshold
            )
            
            if best_bootsize is None:
                continue
            
            # Compute k for this individual subset
            b_area = best_bootsize[0] * best_bootsize[1]
            log_n = np.log(n_subset)
            log_b = np.log(b_area)
            k = log_n / log_b
            
            # Store individual result
            all_k_data.append({
                'dim_y': dim_y,
                'dim_x': dim_x,
                'n_subset': n_subset,
                'log_n': log_n,
                'b_opt': best_bootsize,
                'b_area': b_area,
                'log_b': log_b,
                'k': k,
                'error': best_error,
                'location': (idy, idx)
            })
            
            valid_count += 1
            pbar.update(1)
        
        pbar.close()
        print(f"  Completed {valid_count} valid subsets")
    
    print(f"\nTotal samples collected: {len(all_k_data)}")
    
    return all_k_data


# Run the analysis

In [6]:
all_k_data = run_multiscale_analysis(ds, SUBSET_SIZES, 
                                    n_subsets_per_size=N_SUBSETS_PER_SIZE,
                                    error_threshold=ERROR_THRESHOLD)


Multi-scale analysis with 5 subset sizes
Sampling 10 subsets per size
Total samples: 50
Error threshold: 5.0%

Processing subset size: 64 x 45


  Size 64x45:   0%|                                      | 0/10 [00:00<?, ?it/s]

Dimensions ('y', 'x') are already in the expected order
Dimension y has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Dimension x has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Using bootsize: {'y': 64, 'x': 45}
Bootstrappable dimensions: []
No bootstrappable dimensions available. Structure functions will be calculated using the full dataset without bootstrapping or spacings.

STARTING ISOTROPIC_SF WITH FUNCTION TYPE: longitudinal
Variables: ['u', 'v'], Order: 2
Confidence level: 0.95


No bootstrappable dimensions available. Calculating structure function once with full dataset.
Dimensions ('y', 'x') are already in the expected order
Using bootsize: {'y': np.int64(4), 'x': np.int64(2)}
Bootstrappable dimensions: ['y', 'x']
Two bootstrappable dimensions. Available spacings: [1, 2, 4, 8, 16]

STARTING ISOTROPIC_SF WITH FUNCTION TYPE: longitudinal
Variables: ['u', 'v'], Orde

  Size 64x45:  10%|███                           | 1/10 [00:08<01:20,  8.92s/it]


CALCULATING BIN DENSITIES
Total points collected: 8791200
Bins with points: 7/11
Marked 4 bins as converged (low_density)
Marked 7 bins as converged (converged_eps)

STARTING ADAPTIVE CONVERGENCE LOOP
All bins have converged or reached max bootstraps!

FINAL CONVERGENCE STATISTICS:
  Total bins with data (>10 points): 7
  Converged bins: 7
  Unconverged bins: 0
  Bins at max bootstraps: 7

Creating output dataset...
ISOTROPIC SF COMPLETED SUCCESSFULLY!
Dimensions ('y', 'x') are already in the expected order
Dimension y has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Dimension x has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Using bootsize: {'y': 64, 'x': 45}
Bootstrappable dimensions: []
No bootstrappable dimensions available. Structure functions will be calculated using the full dataset without bootstrapping or spacings.

STARTING ISOTROPIC_SF WITH FUNCTION TYPE: longi

...


CALCULATING BIN DENSITIES
Total points collected: 2043289174
Bins with points: 9/11
Marked 2 bins as converged (low_density)
Marked 9 bins as converged (converged_eps)

STARTING ADAPTIVE CONVERGENCE LOOP
All bins have converged or reached max bootstraps!

FINAL CONVERGENCE STATISTICS:
  Total bins with data (>10 points): 9
  Converged bins: 9
  Unconverged bins: 0
  Bins at max bootstraps: 9

Creating output dataset...
ISOTROPIC SF COMPLETED SUCCESSFULLY!
Dimensions ('y', 'x') are already in the expected order
Dimension y has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Dimension x has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Using bootsize: {'y': 256, 'x': 180}
Bootstrappable dimensions: []
No bootstrappable dimensions available. Structure functions will be calculated using the full dataset without bootstrapping or spacings.

STARTING ISOTROPIC_SF WITH FUNCTION TYPE: 

  Size 256x180:  90%|█████████████████████████▏  | 9/10 [08:19<00:55, 55.23s/it]


CALCULATING BIN DENSITIES
Total points collected: 2147770080
Bins with points: 9/11
Marked 2 bins as converged (low_density)
Marked 9 bins as converged (converged_eps)

STARTING ADAPTIVE CONVERGENCE LOOP
All bins have converged or reached max bootstraps!

FINAL CONVERGENCE STATISTICS:
  Total bins with data (>10 points): 9
  Converged bins: 9
  Unconverged bins: 0
  Bins at max bootstraps: 9

Creating output dataset...
ISOTROPIC SF COMPLETED SUCCESSFULLY!
Dimensions ('y', 'x') are already in the expected order
Dimension y has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Dimension x has bootsize equal to or larger than dimension size. No bootstrapping will be done across this dimension.
Using bootsize: {'y': 256, 'x': 180}
Bootstrappable dimensions: []
No bootstrappable dimensions available. Structure functions will be calculated using the full dataset without bootstrapping or spacings.

STARTING ISOTROPIC_SF WITH FUNCTION TYPE: 

  Size 256x180: 100%|███████████████████████████| 10/10 [09:14<00:00, 55.43s/it]


CALCULATING BIN DENSITIES
Total points collected: 2147770080
Bins with points: 9/11
Marked 2 bins as converged (low_density)
Marked 9 bins as converged (converged_eps)

STARTING ADAPTIVE CONVERGENCE LOOP
All bins have converged or reached max bootstraps!

FINAL CONVERGENCE STATISTICS:
  Total bins with data (>10 points): 9
  Converged bins: 9
  Unconverged bins: 0
  Bins at max bootstraps: 9

Creating output dataset...
ISOTROPIC SF COMPLETED SUCCESSFULLY!
  Completed 10 valid subsets

Total samples collected: 50


# Compute k statistics Function

In [13]:
def compute_k_statistics(all_k_data, full_dim_y, full_dim_x, subset_sizes):
    """
    Compute scaling exponent k and optimal bootsize using ratio-based scaling.
    Now returns a GLOBAL optimal bootsize that works across all scales.
    """
    
    unique_scales = sorted(set((d['dim_y'], d['dim_x']) for d in all_k_data), 
                           key=lambda x: x[0]*x[1])
    
    log_n_all = np.array([d['log_n'] for d in all_k_data])
    log_b_all = np.array([d['log_b'] for d in all_k_data])
    
    n_full = full_dim_y * full_dim_x
    aspect_ratio = full_dim_y / full_dim_x
    
    if np.round(np.var(log_n_all)) == 0.:
        k_values = log_n_all / log_b_all
        k_robust = np.nanmedian(k_values)
        # Use median of all b values directly
        b_opt_full_area = np.median([d['b_area'] for d in all_k_data])
        slope, intercept, r_squared = None, None, None
        
    else:
        # Regression gives a GLOBAL fit across all scales
        slope = np.cov(log_n_all, log_b_all)[0, 1] / np.var(log_n_all)
        intercept = np.mean(log_b_all) - slope * np.mean(log_n_all)
        k_robust = 1 / slope
        
        # R² calculation
        log_b_pred = slope * log_n_all + intercept
        ss_res = np.sum((log_b_all - log_b_pred) ** 2)
        ss_tot = np.sum((log_b_all - np.mean(log_b_all)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # Global optimal: use regression to predict at full scale
        # This inherently averages information from ALL scales
        log_b_opt_full = slope * np.log(n_full) + intercept
        b_opt_full_area = np.exp(log_b_opt_full)
    
    # Split into dimensions preserving aspect ratio
    b_opt_x = int(round(np.sqrt(b_opt_full_area / aspect_ratio)))
    b_opt_y = int(round(aspect_ratio * b_opt_x))
    
    # Also compute per-scale optimal for diagnostics
    per_scale_b_opt = {}
    for scale in unique_scales:
        b_vals = [d['b_area'] for d in all_k_data 
                  if d['dim_y'] == scale[0] and d['dim_x'] == scale[1]]
        per_scale_b_opt[scale] = np.median(b_vals)
    
    results = {
        'k_robust': k_robust,
        'n_full': n_full,
        'aspect_ratio': aspect_ratio,
        'b_opt_area': b_opt_full_area,
        'b_opt_y': b_opt_y,
        'b_opt_x': b_opt_x,
        'log_n': log_n_all,
        'log_b': log_b_all,
        'per_scale_b_opt': per_scale_b_opt,  # for diagnostics
    }
    
    if slope is not None:
        results.update({
            'slope': slope,
            'intercept': intercept,
            'r_squared': r_squared,
        })
    
    return results

# Run Statistical Analysis

In [15]:
# Run analysis
results = compute_k_statistics(all_k_data, FULL_DIM_Y, FULL_DIM_X, SUBSET_SIZES)



# Summary and Final Recommendations 

In [18]:
def find_pyturbo_compatible_bootsize(b_opt_y, b_opt_x, full_dim_y, full_dim_x):
    """
    Find the closest bootsize that satisfies PyTurbo_SF requirement:
    full_dim / bootsize = 2^k (i.e., bootsize divides full_dim into a power of 2)
    
    Parameters:
    -----------
    b_opt_y, b_opt_x : int
        Computed optimal bootsize dimensions
    full_dim_y, full_dim_x : int
        Full dataset dimensions
        
    Returns:
    --------
    b_compat_y, b_compat_x : int
        PyTurbo-compatible bootsize dimensions
    n_blocks_y, n_blocks_x : int
        Number of blocks (should be powers of 2)
    """
    
    def closest_power_of_2_divisor(b_opt, full_dim):
        """Find b such that full_dim/b is closest power of 2 to full_dim/b_opt."""
        # Target number of blocks
        target_n_blocks = full_dim / b_opt
        
        # Find closest power of 2
        log2_target = np.log2(target_n_blocks)
        
        # Check floor and ceiling powers
        power_floor = int(np.floor(log2_target))
        power_ceil = int(np.ceil(log2_target))
        
        # Corresponding n_blocks and b values
        candidates = []
        for power in [power_floor, power_ceil]:
            if power >= 1:  # At least 2 blocks
                n_blocks = 2 ** power
                b = full_dim / n_blocks
                # b should be integer (or close to it)
                if b >= 1:
                    candidates.append((power, n_blocks, b))
        
        # Pick the one where b is closest to b_opt
        best = min(candidates, key=lambda x: abs(x[2] - b_opt))
        
        return int(best[2]), best[1], best[0]  # b, n_blocks, power
    
    b_compat_y, n_blocks_y, power_y = closest_power_of_2_divisor(b_opt_y, full_dim_y)
    b_compat_x, n_blocks_x, power_x = closest_power_of_2_divisor(b_opt_x, full_dim_x)
    
    return {
        'b_compat_y': b_compat_y,
        'b_compat_x': b_compat_x,
        'n_blocks_y': n_blocks_y,
        'n_blocks_x': n_blocks_x,
        'power_y': power_y,
        'power_x': power_x,
    }


# Compute PyTurbo-compatible bootsize
pyturbo_compat = find_pyturbo_compatible_bootsize(
    results['b_opt_y'], results['b_opt_x'], FULL_DIM_Y, FULL_DIM_X
)

# Display results
print()
print("╔" + "═" * 62 + "╗")
print("║" + " " * 18 + "FINAL RECOMMENDATION" + " " * 24 + "║")
print("╠" + "═" * 62 + "╣")
print(f"║  Method: Ratio-based scaling                                ║")
print(f"║  Full dataset dimensions:  {FULL_DIM_Y} x {FULL_DIM_X}                       ║")
print(f"║  Number of samples:        {len(all_k_data):<32}║")
print("╠" + "═" * 62 + "╣")
print(f"║  k (optimal):              {results['k_robust']:<34.4f}║")
try:
    print(f"║  R²:                       {results['r_squared']:<34.4f}║")
except:
    pass  
print("╠" + "═" * 62 + "╣")
print(f"║  COMPUTED OPTIMAL:         {results['b_opt_y']} x {results['b_opt_x']:<26}║")
print("╠" + "═" * 62 + "╣")
print(f"║  >>> PYTURBO-COMPATIBLE:   {pyturbo_compat['b_compat_y']} x {pyturbo_compat['b_compat_x']:<26}║")
print("╚" + "═" * 62 + "╝")



╔══════════════════════════════════════════════════════════════╗
║                  FINAL RECOMMENDATION                        ║
╠══════════════════════════════════════════════════════════════╣
║  Method: Ratio-based scaling                                ║
║  Full dataset dimensions:  2820 x 2000                       ║
║  Number of samples:        50                              ║
╠══════════════════════════════════════════════════════════════╣
║  k (optimal):              1.8744                            ║
║  R²:                       0.2452                            ║
╠══════════════════════════════════════════════════════════════╣
║  COMPUTED OPTIMAL:         124 x 88                        ║
╠══════════════════════════════════════════════════════════════╣
║  >>> PYTURBO-COMPATIBLE:   88 x 62                        ║
╚══════════════════════════════════════════════════════════════╝
